# Fine-tuning: Test-subset: BCI Competition III Data II (Subj B)

## Цель эксперимента

Провести финальную честную оценку стратегий обучения на **Test-subject B BCI Competition III**, не использовавшейся для выбора гиперпараметров и стратегий.

Сравниваются:

- scratch
- full fine-tuning (`full_ft`)
- low LR encoder (`low_lr_encoder`)
- partial fine-tuning (`partial_ft`)
- warmup fine-tuning (`warmup`)

---

## Фиксированные условия

Все эксперименты проводятся при одинаковых настройках:

- `lr_encoder = 3e-5`
- `lr_head = 3e-4`
- `weight_decay = 1e-3`
- `warmup_epochs = 3`

Важно:

- гиперпараметры **не подбираются**
- split’ы фиксированы
- нормализация считается только по `Calib_p` конкретного субъекта
- `test_rest` не используется при обучении

---

## Протокол

Для каждого субъекта и каждого уровня калибровки

\[
p \in \{0, 10, 20, 40, 60, 100\}
\]

выполняется:

1. формирование `Calib_p`
2. `train/val` split внутри `Calib_p` (для `p > 0`)
3. обучение модели
4. early stopping по `val_loss`
5. оценка на фиксированном `test_rest`

---

## Метрики

Сохраняются:

- ROC-AUC
- Accuracy
- Binary F1
- Precision
- Recall
- FDR

---

## Архитектура

- Encoder: `UNet1DEncoder`
- Head: Global Average Pooling по времени + Linear (512 → 2)
- Loss: CrossEntropy

---

## Важно про Test

Этот ноутбук используется для **финальной оценки**.  
Результаты из него будут использоваться для итоговых графиков, статистического сравнения и выводов в дипломе.

## 1. Импорты

In [1]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import stage5_utils as u

import model_unet as m

## 2. Конфиги

In [2]:
RUNTIME_CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,
}

In [3]:
PATHS = {
    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_results"),
}

## 3. Воспроизводимость

In [4]:
u.set_seed(RUNTIME_CONFIG["seed"])
print("Device:", RUNTIME_CONFIG["device"])

Device: cuda


## 4. Пути

In [5]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": PATHS["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": PATHS["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": PATHS["data_root"] / "bcicompiii-ds2",  
}

In [6]:
GROUPS = ["train", "benchmark", "bcicomp3"]

## Конфиг для Subject B

In [14]:
BCICOMP3_TEST_CONFIG = {
    "tag": "stage5_final_eval_bcicomp3_subjB_all_strategies",
    "subjects": ["subjB"],
    "group": "bcicomp3",

    "p_list": [0, 10, 20, 40, 60, 100],
    "ft_scenarios": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],
    "scratch_scenarios": ["scratch"],
    # scratch
    "scratch_lr": 3e-4,
    "scratch_weight_decay": 1e-4,

    # FT
    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}

In [8]:
RUN_DIR = PATHS["results_root"] / BCICOMP3_TEST_CONFIG["tag"]

ARTIFACT_DIRS = {
    "root": RUN_DIR,
    "ft": RUN_DIR / "ft_strategies",
    "scratch": RUN_DIR / "scratch",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
}

for path in ARTIFACT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

RUN_DIR: /kaggle/working/stage5_results/stage5_final_eval_bcicomp3_subjB_all_strategies


## Полный прогон FT-стратегий на Subject B

In [15]:
final_test_ft_df = u.run_many(
    subject_list=BCICOMP3_TEST_CONFIG["subjects"],
    p_list=BCICOMP3_TEST_CONFIG["p_list"],
    scenario_list=["ssl_ft"],
    group=BCICOMP3_TEST_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=ARTIFACT_DIRS["ft"],

    encoder_checkpoint=PATHS["encoder_checkpoint"],
    ft_strategy_list=BCICOMP3_TEST_CONFIG["ft_scenarios"],

    lr_encoder=BCICOMP3_TEST_CONFIG["lr_encoder"],
    lr_head=BCICOMP3_TEST_CONFIG["lr_head"],
    weight_decay=BCICOMP3_TEST_CONFIG["weight_decay"],
    warmup_epochs=BCICOMP3_TEST_CONFIG["warmup_epochs"],

    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_ft_summary_path = ARTIFACT_DIRS["ft"] / "summary.csv"
final_test_ft_df.to_csv(final_test_ft_summary_path, index=False)

print("FT test run finished.")
print("Saved to:", final_test_ft_summary_path)
print("Shape:", final_test_ft_df.shape)
display(final_test_ft_df.head())

Planned runs: 24
[1/24] subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=full_ft | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[2/24] subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=low_lr_encoder | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[3/24] subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=partial_ft | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[4/24] subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=warmup | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[5/24] subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=full_ft | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[6/24] subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=low_lr_encoder | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[7/24] subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=partial_ft | seed=42 | lr_enc=3e-05 | lr_head=0.0003 | wd=0.001
[8

,subject_id,group,p,scenario,ft_strategy,seed,encoder_checkpoint,lr_encoder,lr_head,weight_decay,...,accuracy,f1,precision,recall,fdr,run_tag,history_path,predictions_path,status,error
0,subjB,bcicomp3,0,ssl_ft,full_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,0.166521,0.2855,0.166521,1.0,0.001064,bcicomp3__subjB__p0__ssl_ft__seed42__full_ft__...,None,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
1,subjB,bcicomp3,0,ssl_ft,low_lr_encoder,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,0.166521,0.2855,0.166521,1.0,0.001064,bcicomp3__subjB__p0__ssl_ft__seed42__low_lr_en...,None,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
2,subjB,bcicomp3,0,ssl_ft,partial_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,0.166521,0.2855,0.166521,1.0,0.001064,bcicomp3__subjB__p0__ssl_ft__seed42__partial_f...,None,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
3,subjB,bcicomp3,0,ssl_ft,warmup,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,0.166521,0.2855,0.166521,1.0,0.001064,bcicomp3__subjB__p0__ssl_ft__seed42__warmup__l...,None,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
4,subjB,bcicomp3,10,ssl_ft,full_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,0.166521,0.2855,0.166521,1.0,0.000024,bcicomp3__subjB__p10__ssl_ft__seed42__full_ft_...,/kaggle/working/stage5_results/stage5_final_ev...,/kaggle/working/stage5_results/stage5_final_ev...,ok,None


## Полный прогон Scratch на Subject B

In [16]:
final_test_scratch_df = u.run_many(
    subject_list=BCICOMP3_TEST_CONFIG["subjects"],
    p_list=BCICOMP3_TEST_CONFIG["p_list"],
    scenario_list=["scratch"],
    group=BCICOMP3_TEST_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=ARTIFACT_DIRS["scratch"],

    scratch_lr=BCICOMP3_TEST_CONFIG["scratch_lr"],
    scratch_weight_decay=BCICOMP3_TEST_CONFIG["scratch_weight_decay"],

    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_scratch_summary_path = ARTIFACT_DIRS["scratch"] / "summary.csv"
final_test_scratch_df.to_csv(final_test_scratch_summary_path, index=False)

print("Scratch test run finished.")
print("Saved to:", final_test_scratch_summary_path)
print("Shape:", final_test_scratch_df.shape)
display(final_test_scratch_df.head())

Planned runs: 6
[1/6] subject=subjB | group=bcicomp3 | p=0 | scenario=scratch | ft_strategy=None | seed=42 | lr_enc=None | lr_head=None | wd=None
[2/6] subject=subjB | group=bcicomp3 | p=10 | scenario=scratch | ft_strategy=None | seed=42 | lr_enc=None | lr_head=None | wd=None
[3/6] subject=subjB | group=bcicomp3 | p=20 | scenario=scratch | ft_strategy=None | seed=42 | lr_enc=None | lr_head=None | wd=None
[4/6] subject=subjB | group=bcicomp3 | p=40 | scenario=scratch | ft_strategy=None | seed=42 | lr_enc=None | lr_head=None | wd=None
[5/6] subject=subjB | group=bcicomp3 | p=60 | scenario=scratch | ft_strategy=None | seed=42 | lr_enc=None | lr_head=None | wd=None
[6/6] subject=subjB | group=bcicomp3 | p=100 | scenario=scratch | ft_strategy=None | seed=42 | lr_enc=None | lr_head=None | wd=None
Scratch test run finished.
Saved to: /kaggle/working/stage5_results/stage5_final_eval_bcicomp3_subjB_all_strategies/scratch/summary.csv
Shape: (6, 33)


,subject_id,group,p,scenario,ft_strategy,seed,encoder_checkpoint,lr_encoder,lr_head,weight_decay,...,accuracy,f1,precision,recall,fdr,run_tag,history_path,predictions_path,status,error
0,subjB,bcicomp3,0,scratch,None,42,None,None,None,NaN,...,0.833479,0.000000,0.000000,0.000000,0.001954,bcicomp3__subjB__p0__scratch__seed42__lre1em05...,None,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
1,subjB,bcicomp3,10,scratch,None,42,None,None,None,0.0001,...,0.426380,0.278788,0.176307,0.665789,0.005134,bcicomp3__subjB__p10__scratch__seed42__lre1em0...,/kaggle/working/stage5_results/stage5_final_ev...,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
2,subjB,bcicomp3,20,scratch,None,42,None,None,None,0.0001,...,0.168712,0.284960,0.166300,0.994737,0.003594,bcicomp3__subjB__p20__scratch__seed42__lre1em0...,/kaggle/working/stage5_results/stage5_final_ev...,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
3,subjB,bcicomp3,40,scratch,None,42,None,None,None,0.0001,...,0.446757,0.279601,0.178506,0.644737,0.002710,bcicomp3__subjB__p40__scratch__seed42__lre1em0...,/kaggle/working/stage5_results/stage5_final_ev...,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
4,subjB,bcicomp3,60,scratch,None,42,None,None,None,0.0001,...,0.433392,0.335560,0.208493,0.859211,0.174734,bcicomp3__subjB__p60__scratch__seed42__lre1em0...,/kaggle/working/stage5_results/stage5_final_ev...,/kaggle/working/stage5_results/stage5_final_ev...,ok,None
